In [4]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
import jieba
import numpy as np
from sklearn.impute import SimpleImputer

# 特征抽取
 当特征中含有字符串时(当作类别)

In [5]:
def dictvec():
    """
    字典数据抽取
    :return: 
    """
    dict1 = DictVectorizer(sparse=False)
    # sparse=True, 则是改为稀疏矩阵存储, 但是稀疏矩阵不能直接喂给模型
    # 用于字典的矢量化器
    data = dict1.fit_transform([{'city': '北京', 'temperature': 100},
                                {'city': '上海', 'temperature': 60},
                                {'city': '深圳', 'temperature': 30}])
    # 将字符串的类别数据变成了One-Hot编码, 模型不关心特征名
    print(data)
    print('-' * 50)
    # 字典中的一些类别数据，分别进行转换成特征
    print(dict1.get_feature_names_out())
    # 特征名
    print('-' * 50)
    print(dict1.inverse_transform(data))  
    # 每个特征代表的含义


dictvec()

[[  0.   1.   0. 100.]
 [  1.   0.   0.  60.]
 [  0.   0.   1.  30.]]
--------------------------------------------------
['city=上海' 'city=北京' 'city=深圳' 'temperature']
--------------------------------------------------
[{'city=北京': np.float64(1.0), 'temperature': np.float64(100.0)}, {'city=上海': np.float64(1.0), 'temperature': np.float64(60.0)}, {'city=深圳': np.float64(1.0), 'temperature': np.float64(30.0)}]


将英文文本转换为数值类型

In [7]:
def couvec():
    # min_df：最小文档频率
    # int：词至少出现在多少个文档中(而不是出现次数)
    # float：词至少占总文档数的比例（如0.01表示1%）
    # max_df：最大文档频率
    # int：词最多允许出现在多少个文档中
    # float：词最多占总文档数的比例（如0.8表示80%）

    vector = CountVectorizer(min_df=2)
    # 与上个例子相同, 先创建对象
    # CountVectorizer指的是统计词频的矢量化器
    # min_df=2意为至少出现在两个文档里
    # 默认会去除单个字母的单词，默认认为这个词对整个样本没有影响,认为其没有语义
    
    res = vector.fit_transform(
        ["life is short,i like python life",
         "life is too long,i dislike python",
         "life is short"])
    # 这里有三个字符串(三个文档)
    # 这里的fit_transform()方法，将文本转为矩阵，返回稀疏矩阵
    
    print(res)
    print(type(res))
    print(vector.get_feature_names_out())
    # 特征名(词汇表)
    print('-' * 50)
    print(res.toarray())  
    # 转换为数组
    print('-' * 50)
    print(vector.inverse_transform(res)) 
    # 反推文档包含的词
    

couvec()

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 10 stored elements and shape (3, 4)>
  Coords	Values
  (0, 1)	2
  (0, 0)	1
  (0, 3)	1
  (0, 2)	1
  (1, 1)	1
  (1, 0)	1
  (1, 2)	1
  (2, 1)	1
  (2, 0)	1
  (2, 3)	1
<class 'scipy.sparse._csr.csr_matrix'>
['is' 'life' 'python' 'short']
--------------------------------------------------
[[1 2 1 1]
 [1 1 1 0]
 [1 1 0 1]]
--------------------------------------------------
[array(['life', 'is', 'short', 'python'], dtype='<U6'), array(['life', 'is', 'python'], dtype='<U6'), array(['life', 'is', 'short'], dtype='<U6')]


数值化中文文本

In [8]:

def countvec():
    """
    CountVectorizer只会粗分中文文本
    :return:
    """
    cv = CountVectorizer()

    data = cv.fit_transform(["人生苦短，我喜欢 python python",
                             "人生漫长，不用 python"])

    print(cv.get_feature_names_out())
    print('-' * 50)
    print(data) 
    # 稀疏矩阵
    print('-' * 50)
    print(data.toarray())


countvec()

['python' '不用' '人生漫长' '人生苦短' '我喜欢']
--------------------------------------------------
<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 6 stored elements and shape (2, 5)>
  Coords	Values
  (0, 3)	1
  (0, 4)	1
  (0, 0)	2
  (1, 0)	1
  (1, 2)	1
  (1, 1)	1
--------------------------------------------------
[[2 0 0 1 1]
 [1 1 1 0 0]]


In [12]:
def cutword():
    """
    通过jieba对中文进行分词
    :return:
    """
    con1 = jieba.cut("今天很残酷，明天更残酷，后天很美好，但绝对大部分是死在明天晚上，所以每个人不要放弃今天。")

    con2 = jieba.cut("我们看到的从很远星系来的光是在几百万年之前发出的，这样当我们看到宇宙时，我们是在看它的过去。")

    con3 = jieba.cut("如果只用一种方式了解某样事物，你就不会真正了解它。了解事物真正含义的秘密取决于如何将其与我们所了解的事物相联系。")
    # cut方法返回的是一个生成器

    print(type(con1))
    print('-' * 50)
    
    content1 = list(con1)
    content2 = list(con2)
    content3 = list(con3)
    # 把生成器转换成列表
    print(content1)
    print(content2)
    print(content3)
    print('-' * 50)
    
    c1 = ' '.join(content1)
    c2 = ' '.join(content2)
    c3 = ' '.join(content3)
    # 把列表转换成字符串,每个词之间用空格隔开
    return c1, c2, c3
# CountVectorizer只能处理字符串, 所以先用jieba分词, 转换成带空格的字符串

def chinese_vec():
    """
    中文特征值化
    :return: None
    """
    c1, c2, c3 = cutword()  
    #带空格的字符串
    print('-' * 50)
    
    print(c1)  
    print(c2)
    print(c3)
    print('-' * 50)
    
    cv = CountVectorizer()
    data = cv.fit_transform([c1, c2, c3])
    print(cv.get_feature_names_out()) 
    # 有了空格就能正常处理中文了
    # 忽略单个汉字组成的词, 认为不影响整体
    print(data.toarray())

    return None


# cutword()
chinese_vec()

<class 'generator'>
--------------------------------------------------
['今天', '很', '残酷', '，', '明天', '更', '残酷', '，', '后天', '很', '美好', '，', '但', '绝对', '大部分', '是', '死', '在', '明天', '晚上', '，', '所以', '每个', '人', '不要', '放弃', '今天', '。']
['我们', '看到', '的', '从', '很', '远', '星系', '来', '的', '光是在', '几百万年', '之前', '发出', '的', '，', '这样', '当', '我们', '看到', '宇宙', '时', '，', '我们', '是', '在', '看', '它', '的', '过去', '。']
['如果', '只用', '一种', '方式', '了解', '某样', '事物', '，', '你', '就', '不会', '真正', '了解', '它', '。', '了解', '事物', '真正', '含义', '的', '秘密', '取决于', '如何', '将', '其', '与', '我们', '所', '了解', '的', '事物', '相', '联系', '。']
--------------------------------------------------
--------------------------------------------------
今天 很 残酷 ， 明天 更 残酷 ， 后天 很 美好 ， 但 绝对 大部分 是 死 在 明天 晚上 ， 所以 每个 人 不要 放弃 今天 。
我们 看到 的 从 很 远 星系 来 的 光是在 几百万年 之前 发出 的 ， 这样 当 我们 看到 宇宙 时 ， 我们 是 在 看 它 的 过去 。
如果 只用 一种 方式 了解 某样 事物 ， 你 就 不会 真正 了解 它 。 了解 事物 真正 含义 的 秘密 取决于 如何 将 其 与 我们 所 了解 的 事物 相 联系 。
--------------------------------------------------
['一种' '不会' '不要' '之前' 

TF-IDF

In [13]:
# 为了解决"停用词", 即"的", "所以"之类的特别常见的词, 它们不能帮助判断文档的类别
# TF-IDF的主要思想是:如果一个词在某个文档出现频率高, 但在其他文档中很少出现, 则认为适合用来分类

def tfidf_vec():
    """
    中文特征值化,计算tfidf值
    :return: None
    """
    c1, c2, c3 = cutword()

    print(c1, c2, c3)
    print(type([c1, c2, c3]))
    tf = TfidfVectorizer(smooth_idf=True)
    # 计算TF*IDF
    # TF: 这个词在当前文档中的词频; 
    # IDF: 这个词在语料库中的逆文档频率, log(总文档数量/包含该词的文档数量)
    # 常见词的IDF低, 稀有词的IDF高
    # smooth_idf: 防止IDF值为0, 即某词出现在所有文档中, log算出来为0
    # 公式log((N+1)/(df+1))+1

    data = tf.fit_transform([c1, c2, c3])

    print(tf.get_feature_names_out())
    print('-' * 50)
    print(type(data))
    print('-' * 50)
    print(data.toarray())
    # 结果相当于上面的chinese_vec()数组乘了一个IDF

    return None


tfidf_vec()

<class 'generator'>
--------------------------------------------------
['今天', '很', '残酷', '，', '明天', '更', '残酷', '，', '后天', '很', '美好', '，', '但', '绝对', '大部分', '是', '死', '在', '明天', '晚上', '，', '所以', '每个', '人', '不要', '放弃', '今天', '。']
['我们', '看到', '的', '从', '很', '远', '星系', '来', '的', '光是在', '几百万年', '之前', '发出', '的', '，', '这样', '当', '我们', '看到', '宇宙', '时', '，', '我们', '是', '在', '看', '它', '的', '过去', '。']
['如果', '只用', '一种', '方式', '了解', '某样', '事物', '，', '你', '就', '不会', '真正', '了解', '它', '。', '了解', '事物', '真正', '含义', '的', '秘密', '取决于', '如何', '将', '其', '与', '我们', '所', '了解', '的', '事物', '相', '联系', '。']
--------------------------------------------------
今天 很 残酷 ， 明天 更 残酷 ， 后天 很 美好 ， 但 绝对 大部分 是 死 在 明天 晚上 ， 所以 每个 人 不要 放弃 今天 。 我们 看到 的 从 很 远 星系 来 的 光是在 几百万年 之前 发出 的 ， 这样 当 我们 看到 宇宙 时 ， 我们 是 在 看 它 的 过去 。 如果 只用 一种 方式 了解 某样 事物 ， 你 就 不会 真正 了解 它 。 了解 事物 真正 含义 的 秘密 取决于 如何 将 其 与 我们 所 了解 的 事物 相 联系 。
<class 'list'>
['一种' '不会' '不要' '之前' '了解' '事物' '今天' '光是在' '几百万年' '发出' '取决于' '只用' '后天' '含义'
 '大部分' '如何' '如果' '宇宙' '我们' '所以' '

# 特征处理, 统一量纲. 
是对每列(每项特征值)进行缩放

In [14]:
def mm():
    """
    归一化
    :return: None
    """
    mm1 = MinMaxScaler(feature_range=(0, 1))
    # feature_range表示归一化后的特征值范围，默认是(0,1)

    train_data = mm1.fit_transform([[90, 2, 10, 40], 
                                    [60, 4, 15, 45], 
                                    [75, 3, 13, 46]])
    # 四个特征, 三个样本

    print(train_data)
    print('-' * 50)
    test_data = mm1.transform([[1, 2, 3, 4], [6, 5, 8, 7]]) 
    # 用训练集确定的最小值和最大值对测试集进行归一化
    # x' = (x-min)/(max-min)
    # x''=x'*(mx-mi)+mi 其中(mx, mi)=(0, 1)
    # 将feature_range设置为(-1,1)就可以使mx=1, mi=-1
    print(test_data)
    return None
    # transform和fit_transform不同
    # fit_transform()方法用于训练集, 会记录最大最小值
    # transform()方法用于测试集, 使用训练集记录的参数进行归一化


mm()
# 好处是更容易通过梯度下降找到最优解

[[1.         0.         0.         0.        ]
 [0.         1.         1.         0.83333333]
 [0.5        0.5        0.6        1.        ]]
--------------------------------------------------
[[-1.96666667  0.         -1.4        -6.        ]
 [-1.8         1.5        -0.4        -5.5       ]]


In [19]:
# 归一化容易受极端值的影响, 所以样本量大时采用标准化
def stand():
    """
    标准化. 不会改变分布形状, 只是变为均值为0，方差为1的分布
    :return:
    """
    std = StandardScaler()

    data = std.fit_transform([[1., -1., 3.],
                              [2., 4., 2.],
                              [4., 6., -1.]])

    print(data)
    print(type(data))
    print('-' * 50)
    print(std.mean_)
    # 均值
    print(std.var_)  
    # 方差(使用的就是样本数而不是样本数-1, 不需要估计真实参数)
    print(std.n_samples_seen_)  
    # 样本数
    # 以上三个是std.fit_transform()记录的参数, 用来进行标准化
    return data


data1 = stand()

[[-1.06904497 -1.35873244  0.98058068]
 [-0.26726124  0.33968311  0.39223227]
 [ 1.33630621  1.01904933 -1.37281295]]
<class 'numpy.ndarray'>
--------------------------------------------------
[2.33333333 3.         1.33333333]
[1.55555556 8.66666667 2.88888889]
3.0


In [20]:
std1 = StandardScaler()
# 证明标准化后的结果的均值是0，方差为1
data2 = std1.fit_transform(data1)
# print(data2)  
# 标准化又标准化的结果, 没什么意义

print(std1.mean_)
# 均值, 接近0
print(std1.var_)
# 方差

[-1.48029737e-16  7.40148683e-17  7.40148683e-17]
[1. 1. 1.]


# 处理缺失值

In [21]:
def im():
    """
    缺失值插补
    :return:None
    """
    im1 = SimpleImputer(missing_values=np.nan, strategy='mean')
    # np.nan替换为均值
    # mean平均数, median中位数, most_frequent众数, constant指定值
    data = im1.fit_transform([[1, 2], [np.nan, 3], [7, 6], [3, 2]])
    print(data)
    return None


im()

[[1.         2.        ]
 [3.66666667 3.        ]
 [7.         6.        ]
 [3.         2.        ]]


# 特征选择-降维
使特征数变少, 提高训练速度. 删除冗余特征和噪声特征
可以改变值也可以不改变值

三种主要方式: 
Filter(筛选): VarianceThreshold(方差阈值); 
Wrapper(包装): 正则化, 决策树; 
Embedded(嵌入): 集成学习

In [22]:
# 方差阈值法
def var():
    """
    特征选择-删除低方差的特征
    :return: None
    """
    var1 = VarianceThreshold(threshold=0.1)
    # 默认只删除方差为0的特征
    # threshold是方差阈值，删除比这个值小的特征
    data = var1.fit_transform([[0, 2, 0, 3],
                               [0, 1, 4, 3],
                               [0, 1, 1, 3]])

    print(data)
    print('-' * 50)
    print('The support is %s' % var1.get_support(True))
    # 获取剩余特征的原有列索引(被选中/支持的特征)
    return None


var()

[[2 0]
 [1 4]
 [1 1]]
--------------------------------------------------
The surport is [1 2]


In [27]:
def pca():
    """
    使用主成分分析进行特征降维
    :return: None
    """
    # PCA: Principal Component Analysis
    original_value = np.array([[2, 8, 4, 5],
                               [6, 3, 0, 8],
                               [5, 4, 9, 1]])
    print(np.var(original_value, axis=0).sum())  
    # 原数据每一列的方差和
    print('-' * 50)
    pca1 = PCA(n_components=0.9)
    # n_components: 是0到1之间的浮点数时,意思是希望主成分解释的方差比例
    # 即得到保留的每一列方差值除以原有所有数据的方差
    # 一般选择保留90%到95%
    # 如果是整数,则n_components表示保留的主成分数量

    data = pca1.fit_transform(original_value)
    # data是pca处理后的原数组
    print(data)
    print(type(data))
    print(np.var(data, axis=0).sum())
    # data的总方差
    print('-' * 50)
    
    print(pca1.explained_variance_ratio_)
    # data每列方差解释总方差的比例
    print(pca1.explained_variance_ratio_.sum())
    # data的总方差解释原数据总方差的比例, 恰好几乎完全解释
    # 事实上如果把阈值改成0.5, 则主成分数只有1, 只解释0.75的方差
    return None


pca()

29.333333333333336
--------------------------------------------------
[[-1.28620952e-15  3.82970843e+00]
 [-5.74456265e+00 -1.91485422e+00]
 [ 5.74456265e+00 -1.91485422e+00]]
<class 'numpy.ndarray'>
29.333333333333332
--------------------------------------------------
[0.75 0.25]
1.0
